In [1]:
import pandas as pd
import readability
import os
import numpy as np
import random
import torch
from bert_score import BERTScorer
import re
from nltk.translate import meteor
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from textblob import TextBlob
import math

In [2]:
from transformers import logging
logging.set_verbosity_error() # make sure only important transformers logging output is visible

In [3]:
# Only need to run once.
# nltk.download('all')

In [4]:
# Set random states.
def set_random_states(random_state):
    # Set various random seeds.
    np.random.seed(random_state)
    pd.core.common.random_state(random_state)
    random.seed(random_state)
    torch.manual_seed(random_state)
    torch.cuda.manual_seed_all(random_state)
    os.environ["PYTHONHASHSEED"] = str(random_state)
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass
    return random_state

RANDOM_STATE = set_random_states(1618)

In [5]:
OUTPUT_DIR = 'outputScores/'

In [6]:
def preprocess_text(text):
    text = re.sub(r'[^\w\s/]', '', text)
    # tokenize
    tokens = word_tokenize(text.lower())
    # remove stop words
    filtered_tokens = [token for token in tokens if token not in stopwords.words('english')]
    # lemmatize the tokens
    lemmatizer = WordNetLemmatizer()
    lemmatized_tokens = [lemmatizer.lemmatize(token) for token in filtered_tokens]
    # join the tokens back into a string
    processed_text = ' '.join(lemmatized_tokens)
    return processed_text

In [7]:
generation_types = ['fakeLocal', 'fakeOnlineGPT3', 'fakeOnlineGPT4']
input_prompt_paths = ['./dataGeneration/syntheticNotesLocal/fakeNoteGeneration/fake_notes.xlsx', './dataGeneration/syntheticNotesLocal/fakeNoteGeneration/fake_notes.xlsx', './dataGeneration/syntheticNotesLocal/fakeNoteGeneration/fake_notes.xlsx']
input_generated_data_dirs = ['dataGeneration/syntheticNotesLocal/fakeNoteGeneration/processedFakeNoteSyntheticNotes', 'dataGeneration/syntheticNotesOnline/gpt3Notes', 'dataGeneration/syntheticNotesOnline/gpt4Notes']

In [8]:
def make_score_df(generation_type, input_prompt_path, input_generated_data_dir):
    prompt_df = pd.read_excel(input_prompt_path)
    average_scores_dict = []
    # Make BERT scorer.
    scorer = BERTScorer(model_type="bert-base-uncased")
    for dataset in os.listdir(f"./{input_generated_data_dir}"):
        if dataset.endswith('.csv'):
            temp_df = pd.read_csv(f"./{input_generated_data_dir}/{dataset}")
            temp_split = dataset.split('_')
            prompt_number, prompt_needs = temp_split[0], temp_split[1].replace('.csv', '')
            temp_prompt = prompt_df[prompt_df['Needs'] == prompt_needs]['Note'].tolist()[int(prompt_number)]
            # --------- Readability (number linked to grade of reading level)s ---------
            references = []
            candidates = []
            meteors = []
            sentiments = []
            subjectivities = []
            abs_sentiment_diffs = []
            abs_subjectivity_diffs = []
            # Make readability metric.
            preprocessed_prompt = preprocess_text(temp_prompt)
            reference_blob = TextBlob(preprocessed_prompt)
            for temp_generation in temp_df['report'].tolist():
                processed_generation = preprocess_text(temp_generation)
                candidate_blob = TextBlob(processed_generation)
                if candidate_blob.sentences and reference_blob.sentences:
                    # Make candidates and references without punctuation for metrics (BERTScore penalizes punctuation, we should take this into account - https://aclanthology.org/2023.findings-acl.381.pdf).
                    reference = re.sub(r'[^\w\s/]', '', temp_prompt)
                    candidate = re.sub(r'[^\w\s/]', '', temp_generation)
                    references.append(reference)
                    candidates.append(candidate)
                    sentiments.append(candidate_blob.sentences[0].sentiment.polarity)
                    subjectivities.append(candidate_blob.sentences[0].sentiment.subjectivity)
                    # Make METEOR
                    meteors.append(meteor([word_tokenize(candidate)], word_tokenize(reference)))
                    # Make sentiment and subjectivity absolute differences.                    
                    abs_sentiment_diffs.append(math.sqrt((reference_blob.sentences[0].sentiment.polarity - candidate_blob.sentences[0].sentiment.polarity) ** 2))
                    abs_subjectivity_diffs.append(math.sqrt((reference_blob.sentences[0].sentiment.subjectivity - candidate_blob.sentences[0].sentiment.subjectivity) ** 2))

            # Make BERTScore
            _, _, F1 = scorer.score(candidates, references)

            average_scores_dict.append({
                'dataset': dataset.replace('.csv', ''), 
                'needs': prompt_needs,
                'generation_type': generation_type,
                'number_of_notes': len(temp_df),
                # save the F1 values, as these are recommended for use by the BERTScore authors - http://arxiv.org/abs/1904.09675
                'bertscore': float(F1.mean()),
                'meteor': sum(meteors) / len(meteors),
                'sentiment': sum(sentiments) / len(sentiments),
                'subjectivity': sum(subjectivities) / len(subjectivities),
                'abs_sentiment_diff': sum(abs_sentiment_diffs) / len(abs_sentiment_diffs),
                'abs_subjectivity_diff': sum(abs_subjectivity_diffs) / len(abs_subjectivity_diffs)})
            
    # min-max normalize every score
    resultant_df = pd.DataFrame(average_scores_dict)
    return resultant_df


In [9]:
all_dfs = []
for generation_type, input_prompt_path, input_generated_data_dir in zip(generation_types, input_prompt_paths, input_generated_data_dirs):
    all_dfs.append(make_score_df(generation_type, input_prompt_path, input_generated_data_dir))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [10]:
combined_df = pd.concat(all_dfs)

In [11]:
scaled_df = combined_df.copy()

metric_cols = [
    "bertscore",
    "meteor",
    "abs_sentiment_diff",
    "abs_subjectivity_diff"
]

# Min-max normalize safely.
for col in metric_cols:
    min_val = scaled_df[col].min()
    max_val = scaled_df[col].max()

    if max_val - min_val == 0:
        scaled_df[col] = 0.0
    else:
        scaled_df[col] = (scaled_df[col] - min_val) / (max_val - min_val)

# Invert the lower is better columns.
lower_is_better = [
    "abs_sentiment_diff",
    "abs_subjectivity_diff"
]

for col in lower_is_better:
    scaled_df[col] = 1 - scaled_df[col]

# Get overall scores.
scaled_df["overall_score"] = scaled_df[[
    "bertscore",
    "meteor",
    "abs_sentiment_diff",
    "abs_subjectivity_diff"
]].mean(axis=1)

# Get final summary table.
summary = (
    scaled_df
    .groupby(["generation_type"])["overall_score"]
    .mean()
    .reset_index()
    .sort_values(["overall_score"], ascending=False)
)

# Save outputs. 
os.makedirs(f'./{OUTPUT_DIR}/', exist_ok=True)
scaled_df.to_csv(f'./{OUTPUT_DIR}/evaluated_scaled_results.csv', index=False)
combined_df.to_csv(f'./{OUTPUT_DIR}/raw_evaluation_results.csv', index=False)
summary.to_csv(f'./{OUTPUT_DIR}/dataset_summary.csv', index=False)

In [12]:
print("------ Summary Performance ------")
summary

------ Summary Performance ------


,generation_type,overall_score
0,fakeLocal,0.616773
1,fakeOnlineGPT3,0.546350
2,fakeOnlineGPT4,0.537258


In [13]:
print("------ Scaled Scores ------")
scaled_df.sort_values(by='overall_score', ascending=False)

------ Scaled Scores ------


,dataset,needs,generation_type,number_of_notes,bertscore,meteor,sentiment,subjectivity,abs_sentiment_diff,abs_subjectivity_diff,overall_score
8,4_met,met,fakeOnlineGPT3,588,1.000000,1.000000,0.084496,0.292322,0.726939,0.773028,0.874992
9,1_unmet,unmet,fakeLocal,210,0.901011,0.339117,-0.007766,0.380985,0.968051,1.000000,0.802045
5,3_met,met,fakeOnlineGPT4,636,0.773509,0.591258,0.097503,0.371358,0.828206,0.846005,0.759745
8,4_met,met,fakeLocal,170,0.893242,0.624549,0.086177,0.361225,0.810950,0.649272,0.744503
3,4_unmet,unmet,fakeLocal,241,0.688379,0.584248,0.006554,0.366239,1.000000,0.663719,0.734086
0,0_met,met,fakeOnlineGPT4,628,0.623887,0.516248,0.099817,0.417302,0.845186,0.942322,0.731911
5,3_met,met,fakeOnlineGPT3,570,0.834048,0.651159,0.214944,0.430629,0.650796,0.716521,0.713131
9,1_unmet,unmet,fakeOnlineGPT4,643,0.675712,0.330829,0.047549,0.385867,0.877794,0.868597,0.688233
5,3_met,met,fakeLocal,272,0.602803,0.416450,0.144183,0.424871,0.848556,0.795258,0.665767
0,0_met,met,fakeOnlineGPT3,594,0.562118,0.564981,0.168317,0.435939,0.641064,0.844558,0.653180


In [14]:
print("------ Raw Dataframe ------")
combined_df

------ Raw Dataframe ------


,dataset,needs,generation_type,number_of_notes,bertscore,meteor,sentiment,subjectivity,abs_sentiment_diff,abs_subjectivity_diff
0,0_met,met,fakeLocal,216,0.510298,0.156667,0.195341,0.465887,0.213263,0.152570
1,2_unmet,unmet,fakeLocal,278,0.495886,0.097852,0.022852,0.374026,0.146848,0.206472
2,1_met,met,fakeLocal,228,0.515319,0.144650,0.202675,0.450829,0.263694,0.187374
3,4_unmet,unmet,fakeLocal,241,0.526317,0.188962,0.006554,0.366239,0.102096,0.298363
4,3_unmet,unmet,fakeLocal,238,0.523667,0.134512,0.050599,0.377065,0.119002,0.377065
5,3_met,met,fakeLocal,272,0.517312,0.155138,0.144183,0.424871,0.156578,0.233848
6,2_met,met,fakeLocal,232,0.508511,0.106224,0.208057,0.470148,0.166704,0.254056
7,0_unmet,unmet,fakeLocal,300,0.511271,0.087467,-0.011179,0.382385,0.461846,0.477365
8,4_met,met,fakeLocal,170,0.547876,0.197086,0.086177,0.361225,0.170107,0.305448
9,1_unmet,unmet,fakeLocal,210,0.548694,0.139550,-0.007766,0.380985,0.113590,0.133431


In [15]:
# Global model comparison. 
global_summary = (
    scaled_df
    .groupby("generation_type")[[
        "bertscore",
        "meteor",
        "abs_sentiment_diff",
        "abs_subjectivity_diff",
        "overall_score"
    ]]
    .mean()
    .sort_values("overall_score", ascending=False)
)

In [16]:
print("\n------ Global Model Performance ------\n")
global_summary


------ Global Model Performance ------



,bertscore,meteor,abs_sentiment_diff,abs_subjectivity_diff,overall_score
generation_type,,,,,
fakeLocal,0.633245,0.345371,0.751837,0.736637,0.616773
fakeOnlineGPT3,0.463660,0.399962,0.678779,0.642998,0.546350
fakeOnlineGPT4,0.432679,0.317678,0.699908,0.698765,0.537258
